In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import datetime

sys.path.append("..")

## Connect DB

In [ ]:
from src.setup.setup import setup
from src.config import DATABASE_PATH

setup(DATABASE_PATH.absolute().as_posix())

# Get Data

In [ ]:
import ezyquant as ez
from ezyquant.reader import _SETDataReaderCached
import ezyquant.fields as fld

In [ ]:
sdr = _SETDataReaderCached()
start_date_data = datetime.date(2019, 1, 1)  # ISO format: 2019-01-01
start_date_backtest = datetime.date(2020, 1, 1)  # ISO format: 2020-01-01
end_date = datetime.date.fromisoformat(sdr.last_update())

In [ ]:
ssc = ez.SETSignalCreator(
    start_date=start_date_data.isoformat(),
    end_date=end_date.isoformat(),
    index_list=["BANK"],
)

In [ ]:
close_df = ssc.get_data(field="close", timeframe="daily")
high_df = close_df.copy(deep=True)
low_df = close_df.copy(deep=True)
close_df

In [ ]:
close_df.plot()

## Strategy

In [ ]:
from src.strategies.valuation_factor import get_valuation_metrics, get_score

In [ ]:
pb, pe, ev_div_ebitda, fcf_yield = get_valuation_metrics(ssc)

In [ ]:
ev_div_ebitda

In [ ]:
score_df = get_score(pb, pe, ev_div_ebitda, fcf_yield)

In [ ]:
import pandas as pd

In [ ]:
pct_rank = score_df.rank(axis=1, pct=True)
buy_signal = pd.DataFrame(data=False, index=score_df.index, columns=score_df.columns)
# buy_signal &= (pct_rank.count(axis=1) != 0)
buy_signal

buy_signal[pct_rank >= 0.8] = True

buy_signal

In [ ]:
pb.plot()

In [ ]:
pe.plot()

In [ ]:
pe[pe < 50].plot()

In [ ]:
ev_div_ebitda.plot()

In [ ]:
fcf_yield.plot()

In [ ]:
pe = ssc.get_data(field="pe", timeframe="daily")
pe.head(5)

In [ ]:
median_market_pe = pe.transpose().median()
median_market_pe

In [ ]:
median_market_pe.describe()

In [ ]:
median_market_pe.plot()  # Mean PE spikes in late 2020 due to COVID, so we use median instead

In [ ]:
median_market_pe.rolling(28).mean().plot()

In [ ]:
for date, is_pass_threshold in zip(pe.columns, (pe < 14.7).loc["2023-12-28"]):
    print(f"{date}: {is_pass_threshold}")

In [ ]:
pe.lt(median_market_pe * 0.8, axis=0)

In [ ]:
POS_NUM = 20  # TODO: Remove magic number
signal_bank_df = ssc.rank(factor_df=pe, quantity=POS_NUM, ascending=True)
signal_bank_df

In [ ]:
# TODO: Select 20 companies max for manageable diverse portfolio

In [ ]:
from src.strategies.valuation_factor import get_valuation_metrics

In [ ]:
pb, pe, ev_div_ebitda, fcf_yield = get_valuation_metrics(ssc)

In [ ]:
pb

In [ ]:
from src.strategies.valuation_factor import get_signal

signal_df = get_signal(ssc)

In [ ]:
# signal_df[signal_df.isna() | signal_df.notna()] = 0
signal_df

## Backtest

In [ ]:
from src.strategies.valuation_factor import get_backtest_algorithm

In [ ]:
INITIAL_CASH = 50000
PCT_COMMISSION = 0.15
PCT_SELL_SLIP = 0.01

In [ ]:
result = ez.backtest(
    signal_df=signal_df,
    backtest_algorithm=get_backtest_algorithm(pos_num=20),  # TODO: Remove magic number
    start_date=start_date_backtest.isoformat(),
    end_date=end_date.isoformat(),
    initial_cash=INITIAL_CASH,
    pct_commission=PCT_COMMISSION,
    pct_sell_slip=PCT_SELL_SLIP,
    price_match_mode="weighted",
    signal_delay_bar=1
)

In [ ]:
set_index_df = sdr.get_data_index_daily(
    field=fld.D_INDEX_CLOSE,
    index_list=[fld.MARKET_SET],
    start_date=start_date_data.isoformat(),
    end_date=end_date.isoformat(),
)

set_index_return = set_index_df.pct_change().fillna(0.0)

In [ ]:
result.to_basic(benchmark=set_index_return)

## Some more experiment

In [ ]:
rights_benefit_df = sdr.get_rights_benefit(
    symbol_list=list(close_df.columns),
)

with pd.option_context('display.max_rows', None, 'display.max_columns', None):  
    print(rights_benefit_df)

In [ ]:
rights_benefit_df["sign_date"]

In [ ]:
rights_benefit_df['sign_date'] = pd.to_datetime(rights_benefit_df['sign_date'], errors='coerce')
rights_benefit_df['pay_date']  = pd.to_datetime(rights_benefit_df['pay_date'],  errors='coerce')

rights_benefit_df['date'] = rights_benefit_df['pay_date']

dividend_df = rights_benefit_df[rights_benefit_df['dps'].notna()].copy()          # dividend events
dividend_df = dividend_df.dropna(subset=['date'])   # drop rows without any date
dividend_df = dividend_df.sort_values('date')       # time‑series needs chronological order

dividend_df

In [ ]:
pivot = dividend_df.pivot_table(
            index='date',
            columns='symbol',
            values='dps',
            aggfunc='first'          # in case of duplicate rows (rare)
        )

pivot

In [ ]:
from matplotlib import pyplot as plt

In [ ]:
import matplotlib.dates as mdates
mdates.datestr2num(['10/27/2011', '11/2/2011'])

In [ ]:
plt.figure(figsize=(12, 6))
for date in dividend_df["pay_date"]:
    plt.axvspan(mdates.datestr2num(date.isoformat()), mdates.datestr2num(date.isoformat()) + 28)
plt.plot(close_df)

In [ ]:
pd.to_datetime(rights_benefit_df['pay_date'], errors='coerce')

In [ ]:
close_df.plot()